1) Découvrir les nouveaux match_id (sans télécharger)

In [1]:
import sys
from pathlib import Path

# Notebook peut être lancé depuis n'importe quel cwd (VSCode/Jupyter).
# On retrouve la racine du projet (contient le dossier `src/`) et on l'ajoute au PYTHONPATH.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists()), None)
if ROOT is None:
    raise RuntimeError("Project root not found (missing 'src' folder in parents of cwd).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

PosixPath('/home/ju/Documents/Dev/MachineLearning/Dota-Datas')

In [2]:
from pathlib import Path
from src.dota_data.update import download_new_matches

res = download_new_matches(
    teams_csv=ROOT / "data/teams_to_look.csv",
    processed_dir=ROOT / "data/processed",
    raw_updates_dir=ROOT / "data/raw/updates",
    limit=50,
    max_pages=2,
    timeout_team_matches=60,
    dry_run=True,
)
res

/home/ju/Documents/Dev/MachineLearning/Dota-Datas/src/dota_data/update.py:38: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if "match_id" not in lf.columns:
/home/ju/Documents/Dev/MachineLearning/Dota-Datas/src/dota_data/update.py:55: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if not needed.issubset(set(lf.columns)):


{'run_dir': '/home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/raw/updates/run_20260123_125359',
 'run_metadata_path': '/home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/raw/updates/run_20260123_125359/run_metadata.json',
 'run_result_path': '/home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/raw/updates/run_20260123_125359/run_result.json',
 'downloaded': 0,
 'new_match_ids': 2074,
 'dry_run': True}

2) Télécharger + append parquets (petit batch)

In [3]:
from pathlib import Path
from src.dota_data.update import download_new_matches, apply_raw_run_to_processed

res = download_new_matches(
    teams_csv=ROOT / "data/teams_to_look.csv",
    processed_dir=ROOT / "data/processed",
    raw_updates_dir=ROOT / "data/raw/updates",
    limit=50,
    max_pages=2,
    max_new=25,
    chunk_size=25,
    sleep_match_detail=0.5,
    timeout=90,
    timeout_team_matches=60,
    dry_run=False,
)

apply_raw_run_to_processed(
    run_dir=Path(res["run_dir"]),
    processed_dir=ROOT / "data/processed",
    aliases=ROOT / "data/team_aliases.csv",
)

[chunk 1/1] fetching 25 matches (ids 8660611286..8657668675)
[chunk 1/1] saved 25 matches -> matches_chunk_0000.json


{'run_dir': '/home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/raw/updates/run_20260123_125806',
 'raw_files': 1,
 'raw_matches': 25,
 'processed_paths': {'matches': '/home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/processed/matches.parquet',
  'players': '/home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/processed/players.parquet',
  'objectives': '/home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/processed/objectives.parquet',
  'teamfights': '/home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/processed/teamfights.parquet',
  'extras': '/home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/processed/extras.parquet',
  'series': '/home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/processed/series.parquet'}}

3) Valider rapidement l’append

In [4]:
import polars as pl

matches = pl.read_parquet(ROOT / "data/processed/matches.parquet")
players = pl.read_parquet(ROOT / "data/processed/players.parquet")

matches.select(pl.len(), pl.col("match_id").n_unique())
players.select(pl.len(), pl.col("match_id").n_unique())

len,match_id
u32,u32
141950,14195
